# 21. Ensemble Learning: AdaBoost (Adaptive Boosting)

## Algorithm Category
**Type**: Ensemble Learning - Classification/Regression  
**Complexity**: Medium  
**Use Case**: Sequential boosting that adaptively weights misclassified examples

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the AdaBoost algorithm and boosting principles
- Implement AdaBoost for classification and regression
- Understand how AdaBoost adaptively weights training examples
- Visualize how weak learners combine to form a strong classifier
- Tune hyperparameters (n_estimators, learning_rate)
- Apply AdaBoost to real-world problems

## Historical Context

AdaBoost was developed by Freund and Schapire in 1996:
- Freund, Y. & Schapire, R.E. (1996): "A decision-theoretic generalization of on-line learning"
- First practical boosting algorithm
- Won the Gödel Prize in 2003 for theoretical contributions

**Key Papers/References:**
- Freund, Y. & Schapire, R.E. (1996). "A decision-theoretic generalization of on-line learning"
- Freund, Y. & Schapire, R.E. (1997). "A short introduction to boosting"

## When to Use AdaBoost

AdaBoost is appropriate when:
- You have weak learners (slightly better than random)
- Working with binary or multiclass classification
- You want to improve performance iteratively
- Data has clear patterns that can be learned sequentially
- Interpretability of ensemble is helpful
- Moderate-sized datasets

## Theory & Mechanics

### Mathematical Foundation

AdaBoost combines weak learners sequentially, focusing on misclassified examples.

**Initialization:**
- Start with uniform weights: $w_i^{(1)} = \frac{1}{N}$ for all samples

**For each iteration t = 1, 2, ..., T:**

1. **Train weak learner**: $h_t(x)$ on weighted training data
2. **Calculate error**: $\epsilon_t = \sum_{i=1}^{N} w_i^{(t)} \mathbf{1}(h_t(x_i) \neq y_i)$
3. **Calculate learner weight**: $\alpha_t = \frac{1}{2}\ln\left(\frac{1-\epsilon_t}{\epsilon_t}\right)$
4. **Update sample weights**: 
   $$w_i^{(t+1)} = \frac{w_i^{(t)} \exp(-\alpha_t y_i h_t(x_i))}{Z_t}$$
   Where $Z_t$ is normalization factor

**Final Prediction:**
$$\hat{y} = \text{sign}\left(\sum_{t=1}^{T} \alpha_t h_t(x)\right)$$

### How It Works

1. **Start**: Train first weak learner on uniformly weighted data
2. **Evaluate**: Calculate error and assign weight to learner
3. **Re-weight**: Increase weights of misclassified examples
4. **Repeat**: Train next learner on re-weighted data
5. **Combine**: Weighted majority vote of all learners

### Key Hyperparameters

- **n_estimators**: Number of weak learners (boosting iterations)
- **learning_rate**: Shrinks contribution of each learner (default: 1.0)
- **base_estimator**: Type of weak learner (default: DecisionTreeClassifier with max_depth=1)
- **algorithm**: 'SAMME' or 'SAMME.R' (for multiclass)

### Advantages

- Simple to implement and understand
- No prior knowledge needed about weak learners
- Adaptive: focuses on hard examples
- Less prone to overfitting than single strong learner
- Works well with weak learners (stumps)

### Limitations

- Sensitive to noisy data and outliers
- Sequential training (cannot parallelize)
- May overfit with too many weak learners
- Requires careful tuning of learning rate


## Implementation

Let's implement AdaBoost for classification.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import (
    load_breast_cancer,  # Breast cancer classification dataset
    make_classification  # Generate synthetic classification data
)
from sklearn.ensemble import (
    AdaBoostClassifier,  # AdaBoost for classification
    AdaBoostRegressor  # AdaBoost for regression
)
from sklearn.tree import DecisionTreeClassifier  # Decision trees (weak learners for AdaBoost)
from sklearn.model_selection import (
    train_test_split,  # Split data into train/test sets
    cross_val_score,  # Cross-validation scoring
    GridSearchCV  # Hyperparameter tuning
)
from sklearn.metrics import (
    accuracy_score,  # Calculate accuracy (for classification)
    mean_squared_error  # Calculate MSE (for regression)
)

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.supervised import (
    split_data,  # Split data into train/test sets
    evaluate_classifier,  # Evaluate classification models
    evaluate_regressor  # Evaluate regression models
)
from src.models.classification import (
    calculate_classification_metrics,  # Calculate precision, recall, F1, etc.
    plot_confusion_matrix  # Visualize confusion matrix
)
from src.models.ensemble import extract_feature_importance  # Extract feature importance
from src.utils.benchmarking import benchmark_model_training  # Measure training time
from src.utils.validation import (
    validate_model_output,  # Check if predictions are valid
    check_cross_validation_stability  # Check CV stability
)

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# LOADING THE DATASET: Breast Cancer Classification
# ============================================

# load_breast_cancer() loads the Breast Cancer Wisconsin dataset from scikit-learn
# This is a binary classification problem: predict if tumor is malignant (1) or benign (0)
cancer = load_breast_cancer()  # Returns a Bunch object with data, target, feature_names

# X = Features (inputs): Medical measurements of breast tumors
# cancer.data contains feature values (569 samples × 30 features)
# We convert to DataFrame for easier manipulation
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
# Features include: mean radius, mean texture, mean perimeter, mean area, etc. (30 total)

# y = Target (output): Tumor type (what we want to predict)
# cancer.target contains class labels (0 = benign, 1 = malignant)
y = pd.Series(cancer.target, name='Target')
# 0 = benign (non-cancerous), 1 = malignant (cancerous)

print(f"Dataset Shape: {X.shape}")  # Output: (569, 30) - 569 patients, 30 features
print(f"Classes: {cancer.target_names.tolist()}")  # Output: ['malignant', 'benign']

# ============================================
# TRAIN/TEST SPLIT: Separating Data
# ============================================

# Note: AdaBoost doesn't require feature scaling (uses decision trees internally)
# Decision trees are scale-invariant (splits are based on comparisons, not distances)

# split_data() randomly splits data into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

# ============================================
# MODEL CREATION: AdaBoost Classifier
# ============================================

# AdaBoost combines weak learners (simple models) sequentially
# Each new learner focuses on samples that previous learners got wrong

# Create base estimator (weak learner)
# DecisionTreeClassifier with max_depth=1 is called a "decision stump"
# Decision stumps are very simple (just one split) - perfect weak learners
base_estimator = DecisionTreeClassifier(max_depth=1, random_state=42)
# max_depth=1: Only one split (very simple, slightly better than random)

# Create AdaBoostClassifier
model = AdaBoostClassifier(
    base_estimator=base_estimator,  # Weak learner to use
    n_estimators=50,  # Number of weak learners (boosting iterations)
    #   - More estimators = better but slower
    #   - Typical values: 50-200
    learning_rate=1.0,  # Shrinks contribution of each learner
    #   - 1.0 = full contribution (default)
    #   - < 1.0 = slower learning, more stable
    random_state=42  # Reproducibility
)

# ============================================
# MODEL TRAINING: Sequential Boosting
# ============================================

# .fit() trains the AdaBoost model
# The algorithm:
# 1. Train first weak learner on uniformly weighted data
# 2. Calculate error and assign weight to learner
# 3. Increase weights of misclassified samples
# 4. Train next learner on re-weighted data
# 5. Repeat steps 2-4 for n_estimators iterations
# 6. Combine all learners with weighted majority vote
model.fit(X_train, y_train)  # Train the model

print("\nAdaBoost Classifier:")
print(f"Number of estimators: {model.n_estimators}")  # Number of weak learners (50)
print(f"Learning rate: {model.learning_rate}")  # Learning rate (1.0)

# ============================================
# MAKING PREDICTIONS
# ============================================

# .predict() makes predictions by:
# 1. Each weak learner makes a prediction
# 2. Weight each prediction by learner's weight
# 3. Take weighted majority vote (for classification)
y_pred = model.predict(X_test)  # Class predictions (0 or 1)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)  # Compare predictions to true labels
print(f"\nTest Accuracy: {accuracy:.3f}")  # Display accuracy

# ============================================
# EVALUATING MODEL PERFORMANCE
# ============================================

# Evaluate with helper function
results = evaluate_classifier(model, X_test, y_test)
# Returns dictionary with accuracy and other metrics

# Calculate detailed classification metrics
metrics = calculate_classification_metrics(y_test.values, y_pred)
# Returns dictionary with: accuracy, precision, recall, F1

print(f"Precision: {metrics['precision']:.3f}, Recall: {metrics['recall']:.3f}, F1: {metrics['f1_score']:.3f}")
# Precision: Of predicted positives, how many were actually positive
# Recall: Of actual positives, how many did we catch
# F1: Harmonic mean of precision and recall (balances both)


## Understanding Boosting Process

Let's visualize how AdaBoost improves over iterations.


In [ ]:
# ============================================
# TRACKING PERFORMANCE OVER ITERATIONS: Learning Curve
# ============================================

# AdaBoost improves iteratively - each new weak learner corrects previous mistakes
# We'll track how performance changes as we add more weak learners
# This shows the "learning curve" of the boosting process

# Test different numbers of estimators
n_estimators_range = range(1, 101, 5)  # [1, 6, 11, 16, ..., 96] (every 5)
train_scores = []  # Store training accuracy for each n_estimators
test_scores = []  # Store test accuracy for each n_estimators
estimator_errors = []  # Store average error of weak learners

# Test each number of estimators
for n_est in n_estimators_range:
    # Create AdaBoost with this number of estimators
    ab = AdaBoostClassifier(
        base_estimator=DecisionTreeClassifier(max_depth=1, random_state=42),  # Decision stumps
        n_estimators=n_est,  # Number of weak learners
        learning_rate=1.0,  # Learning rate
        random_state=42  # Reproducibility
    )
    
    # Train the model
    ab.fit(X_train, y_train)
    
    # Calculate accuracies
    train_scores.append(accuracy_score(y_train, ab.predict(X_train)))  # Training accuracy
    test_scores.append(accuracy_score(y_test, ab.predict(X_test)))  # Test accuracy
    
    # Get average error of weak learners (if available)
    if hasattr(ab, 'estimator_errors_'):
        # estimator_errors_: Array of errors for each weak learner
        estimator_errors.append(np.mean(ab.estimator_errors_))  # Average error
        # Lower error = better weak learners

# ============================================
# VISUALIZING LEARNING CURVE
# ============================================

# Create figure with 2 subplots
plt.figure(figsize=(12, 5))  # Figure size: 12×5 inches

# Plot 1: Accuracy vs Number of Estimators
plt.subplot(1, 2, 1)  # 1 row, 2 columns, position 1 (left)

# Plot training and test accuracy
plt.plot(n_estimators_range, train_scores, 'o-', label='Training Accuracy', markersize=3)
plt.plot(n_estimators_range, test_scores, 's-', label='Test Accuracy', markersize=3)
# markersize=3: Small markers (many points)

# Label axes
plt.xlabel('Number of Estimators')  # X-axis: n_estimators
plt.ylabel('Accuracy')  # Y-axis: accuracy (0 to 1)
plt.title('AdaBoost: Performance vs Number of Estimators')  # Chart title
plt.legend()  # Show legend
plt.grid(True, alpha=0.3)  # Add grid

# Plot 2: Weak Learner Error Over Iterations
plt.subplot(1, 2, 2)  # 1 row, 2 columns, position 2 (right)

if estimator_errors:
    # Plot average error of weak learners
    plt.plot(n_estimators_range, estimator_errors, '^-', color='green', markersize=3)
    plt.xlabel('Number of Estimators')  # X-axis: n_estimators
    plt.ylabel('Average Weak Learner Error')  # Y-axis: error (lower is better)
    plt.title('Weak Learner Error Over Iterations')  # Chart title
    plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

# ============================================
# FINDING OPTIMAL NUMBER OF ESTIMATORS
# ============================================

# Find number of estimators with highest test accuracy
optimal_n = n_estimators_range[np.argmax(test_scores)]
print(f"Optimal number of estimators: {optimal_n} with test accuracy: {max(test_scores):.3f}")

# Interpretation:
# - Accuracy usually improves as we add more weak learners
# - Eventually plateaus (diminishing returns)
# - Watch for overfitting: if train accuracy >> test accuracy, too many estimators
# - Weak learner error should decrease over iterations (learners get better)
# - Choose number where test accuracy plateaus


## Validation & Testing

Let's validate the model and compare with single decision tree.


In [ ]:
# ============================================
# VALIDATION 1: Comparing Single Weak Learner vs AdaBoost
# ============================================

# AdaBoost combines multiple weak learners to create a strong learner
# We'll compare AdaBoost (50 weak learners) with a single weak learner
# This demonstrates the power of boosting!

from sklearn.tree import DecisionTreeClassifier  # Decision tree (weak learner)

# ============================================
# TRAINING SINGLE WEAK LEARNER
# ============================================

# Create a single decision stump (same as AdaBoost's base estimator)
dt = DecisionTreeClassifier(max_depth=1, random_state=42)
# max_depth=1: Decision stump (one split only)
# This is a very weak learner (slightly better than random)

# Train the single weak learner
dt.fit(X_train, y_train)  # Train on training data

# Make predictions
dt_pred = dt.predict(X_test)  # Predictions from single weak learner

# Calculate accuracy
dt_accuracy = accuracy_score(y_test, dt_pred)  # Accuracy of single weak learner

# ============================================
# COMPARING PERFORMANCE
# ============================================

print("Comparison: Single Weak Learner vs AdaBoost")
print(f"  Single Decision Stump: {dt_accuracy:.3f}")  # Accuracy of one weak learner
print(f"  AdaBoost (50 stumps): {accuracy:.3f}")  # Accuracy of AdaBoost ensemble
print(f"  Improvement: {accuracy - dt_accuracy:.3f}")  # How much better AdaBoost is

# Interpretation:
# - Single weak learner: Low accuracy (weak by design)
# - AdaBoost ensemble: Much higher accuracy (strong learner)
# - Improvement shows the power of boosting
# - Multiple weak learners combined > single weak learner
# - This demonstrates the "ensemble effect": combining models improves performance

# Validation 2: Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print(f"\nCross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")

stability = check_cross_validation_stability(cv_scores, threshold=0.1)
print(f"  Is Stable: {stability['is_stable']}")

# Validation 3: Check model output
validation_result = validate_model_output(y_pred, y_test.values, task_type='classification')
assert validation_result['valid'], "Invalid predictions!"
assert accuracy > dt_accuracy, "AdaBoost should outperform single weak learner!"
print("\n✓ Validation checks passed")


## Feature Importance

Let's extract feature importance from AdaBoost.


In [ ]:
# ============================================
# FEATURE IMPORTANCE: Understanding What Matters
# ============================================

# AdaBoost provides feature importance based on how often features are used in weak learners
# Features used more often in important weak learners have higher importance
# This helps understand which features the model relies on most

# extract_feature_importance() extracts importance scores from the model
feature_importance = extract_feature_importance(model, feature_names=X.columns.tolist())
# model: The trained AdaBoost model
# feature_names: List of feature names (for labeling)
# Returns: DataFrame with features and their importance scores

print("Top 10 Most Important Features:")
print(feature_importance.head(10))  # Display top 10 features
# Shows which features are most important for classification

# ============================================
# VISUALIZING FEATURE IMPORTANCE
# ============================================

# Create horizontal bar chart
plt.figure(figsize=(10, 6))  # Figure size: 10×6 inches

# Get top 15 features
top_features = feature_importance.head(15)  # Top 15 most important features

# Create horizontal bar plot
plt.barh(range(len(top_features)), top_features['importance'], align='center')
# range(len(top_features)): Y-positions (0, 1, 2, ..., 14)
# top_features['importance']: Bar lengths (importance scores)
# align='center': Center bars on y-positions

# Label y-axis with feature names
plt.yticks(range(len(top_features)), top_features['feature'])
# Y-axis: Feature names (most important at top)

# Label x-axis
plt.xlabel('Importance')  # X-axis: importance score
plt.title('AdaBoost Feature Importance (Top 15)')  # Chart title

# Invert y-axis so most important is at top
plt.gca().invert_yaxis()
# gca(): Get current axes
# invert_yaxis(): Flip y-axis (most important at top)

# Adjust layout
plt.tight_layout()
plt.show()  # Display the plot

# Interpretation:
# - Features at top are most important (used most often in weak learners)
# - Features at bottom are less important (rarely used)
# - This helps identify which medical measurements are most predictive
# - Can guide feature selection or data collection priorities


## Effect of Learning Rate

Let's see how learning rate affects performance.


In [ ]:
# ============================================
# EFFECT OF LEARNING RATE: Controlling Boosting Speed
# ============================================

# Learning rate controls how much each weak learner contributes to the final model
# Lower learning rate = slower learning, more stable (requires more estimators)
# Higher learning rate = faster learning, may overfit (requires fewer estimators)
# We'll test different values to find the optimal balance

# Test different learning rates
learning_rates = [0.1, 0.5, 1.0, 1.5, 2.0]  # Range from conservative to aggressive
lr_train_scores = []  # Store training accuracy for each learning rate
lr_test_scores = []  # Store test accuracy for each learning rate

# Test each learning rate
for lr in learning_rates:
    # Create AdaBoost with this learning rate
    ab = AdaBoostClassifier(
        base_estimator=DecisionTreeClassifier(max_depth=1, random_state=42),  # Decision stumps
        n_estimators=50,  # Same number of estimators for fair comparison
        learning_rate=lr,  # Learning rate to test
        random_state=42  # Reproducibility
    )
    
    # Train the model
    ab.fit(X_train, y_train)  # Train on training data
    
    # Calculate accuracies
    lr_train_scores.append(accuracy_score(y_train, ab.predict(X_train)))  # Training accuracy
    lr_test_scores.append(accuracy_score(y_test, ab.predict(X_test)))  # Test accuracy

# ============================================
# VISUALIZING LEARNING RATE EFFECT
# ============================================

# Create line plot
plt.figure(figsize=(10, 6))  # Figure size: 10×6 inches

# Plot training and test accuracy vs learning rate
plt.plot(learning_rates, lr_train_scores, 'o-', label='Training Accuracy')
plt.plot(learning_rates, lr_test_scores, 's-', label='Test Accuracy')
# X-axis: learning rate, Y-axis: accuracy

# Label axes
plt.xlabel('Learning Rate')  # X-axis: learning rate value
plt.ylabel('Accuracy')  # Y-axis: accuracy (0 to 1)
plt.title('AdaBoost: Effect of Learning Rate')  # Chart title
plt.legend()  # Show legend
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display the plot

# ============================================
# FINDING OPTIMAL LEARNING RATE
# ============================================

# Find learning rate with highest test accuracy
optimal_lr = learning_rates[np.argmax(lr_test_scores)]
# np.argmax(): Index of maximum value
# learning_rates[...]: Get learning rate at that index
print(f"Optimal learning rate: {optimal_lr} with test accuracy: {max(lr_test_scores):.3f}")

# Interpretation:
# - Learning rate = 1.0 is default (full contribution of each learner)
# - Lower learning rate (< 1.0): More stable, less prone to overfitting
# - Higher learning rate (> 1.0): Faster learning, but may overfit
# - Optimal learning rate balances speed and stability
# - If train accuracy >> test accuracy, learning rate may be too high (overfitting)
# - If both are low, learning rate may be too low (underfitting)


## Real-World Application

Let's tune hyperparameters and compare with other ensemble methods.


In [ ]:
# ============================================
# HYPERPARAMETER TUNING: Finding Best Settings
# ============================================

# GridSearchCV tests all combinations of hyperparameters and finds the best
# This is more thorough than testing one parameter at a time

# param_grid defines the hyperparameters to search:
param_grid = {
    'n_estimators': [25, 50, 100],  # Number of weak learners
    # More estimators = better but slower
    
    'learning_rate': [0.5, 1.0, 1.5],  # Learning rate
    # Controls contribution of each learner
    
    'base_estimator__max_depth': [1, 2, 3]  # Depth of weak learners
    # base_estimator__max_depth: Access max_depth of base estimator
    # 1 = decision stump (default), 2-3 = slightly stronger learners
}
# Total combinations: 3 × 3 × 3 = 27 combinations to test

# ============================================
# GRID SEARCH: Testing All Combinations
# ============================================

# GridSearchCV performs cross-validation for each hyperparameter combination
# cv=3: 3-fold cross-validation (reduced from 5 for faster execution)
# scoring='accuracy': Use accuracy to evaluate each combination
# n_jobs=-1: Use all CPU cores (parallel processing, faster)
# verbose=1: Show progress (1 = some output)
grid_search = GridSearchCV(
    AdaBoostClassifier(
        base_estimator=DecisionTreeClassifier(random_state=42),  # Base weak learner
        random_state=42  # Reproducibility
    ),
    param_grid,  # Hyperparameters to search
    cv=3,  # 3-fold cross-validation
    scoring='accuracy',  # Evaluation metric
    n_jobs=-1,  # Parallel processing
    verbose=1  # Show progress
)

# Train and evaluate all combinations
# This may take a while: 27 combinations × 3 folds = 81 models to train
grid_search.fit(X_train, y_train)

# ============================================
# DISPLAYING BEST RESULTS
# ============================================

print("\nBest Hyperparameters:")
print(grid_search.best_params_)  # Best combination found
# Example: {'base_estimator__max_depth': 1, 'learning_rate': 1.0, 'n_estimators': 100}

print(f"Best CV Accuracy: {grid_search.best_score_:.3f}")  # Best cross-validation accuracy

# ============================================
# COMPARING WITH RANDOM FOREST
# ============================================

# Random Forest is another popular ensemble method (uses bagging, not boosting)
# We'll compare AdaBoost (boosting) with Random Forest (bagging)

from sklearn.ensemble import RandomForestClassifier  # Random Forest (bagging ensemble)

# Create Random Forest with similar complexity
rf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
# n_estimators=50: 50 trees (similar to AdaBoost's 50 weak learners)
# max_depth=10: Deeper trees (Random Forest trees are stronger than AdaBoost stumps)

# Train Random Forest
rf.fit(X_train, y_train)  # Train on training data

# Make predictions
rf_pred = rf.predict(X_test)  # Predictions from Random Forest

# Calculate accuracy
rf_accuracy = accuracy_score(y_test, rf_pred)  # Test accuracy

# ============================================
# COMPARISON SUMMARY
# ============================================

print(f"\nComparison with Random Forest:")
print(f"  AdaBoost (tuned): {grid_search.best_score_:.3f} (CV)")  # Cross-validation accuracy
print(f"  Random Forest: {rf_accuracy:.3f} (test)")  # Test accuracy

# Interpretation:
# - AdaBoost: Sequential boosting (learners built one after another)
# - Random Forest: Parallel bagging (trees built independently)
# - Both are ensemble methods but use different strategies
# - Which is better depends on the dataset and problem
# - AdaBoost focuses on hard examples, Random Forest reduces variance
# - Compare CV accuracy (AdaBoost) with test accuracy (Random Forest) - note the difference!


## Summary & Key Takeaways

### Key Concepts Learned

1. **AdaBoost Basics**
   - Sequential ensemble method (boosting)
   - Combines weak learners into strong classifier
   - Adaptively weights misclassified examples
   - Each iteration focuses on previous mistakes

2. **Boosting Process**
   - Start with uniform weights
   - Train weak learner on weighted data
   - Calculate learner weight based on error
   - Re-weight samples (increase weight of misclassified)
   - Repeat until convergence

3. **Key Hyperparameters**
   - **n_estimators**: Number of boosting iterations
   - **learning_rate**: Shrinks contribution of each learner
   - **base_estimator**: Type of weak learner (usually decision stumps)

4. **Best Practices**
   - Use weak learners (decision stumps work well)
   - Tune learning rate to prevent overfitting
   - Monitor training vs test performance
   - Consider early stopping if overfitting occurs

### When to Use AdaBoost

✅ **Good for:**
- Binary and multiclass classification
- When you have weak learners available
- Moderate-sized datasets
- Clear patterns that can be learned sequentially
- When interpretability is helpful

❌ **Not ideal for:**
- Noisy data (sensitive to outliers)
- Very large datasets (sequential training is slow)
- When parallelization is important
- Regression tasks (use AdaBoostRegressor or Gradient Boosting)
- Real-time predictions (can be slow)

### Next Steps

- Try **Gradient Boosting** for more sophisticated boosting
- Explore **XGBoost** for optimized gradient boosting
- Compare with **Random Forest** (bagging vs boosting)
- Use **Early Stopping** to prevent overfitting
